# UrduGuard: honest re-evaluation
Run Cell 1 (upload the CSV), then Cell 2. The dataset has 172 rows but only 92 distinct prompts, so the original split leaks. This notebook re-tests the detector on distinct prompts, on an unseen script and on an unseen attack category.

In [ ]:
# Cell 1: upload the dataset file
# Click 'Choose files' and select urdu_adversarial_dataset.csv
# (it is inside the UrduGuard_Project.zip: urduguard/dataset/urdu_adversarial_dataset.csv)
from google.colab import files
up = files.upload()
CSV = list(up.keys())[0]
print('Loaded:', CSV)


In [ ]:
# Cell 2: honest re-evaluation of the UrduGuard detector
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split

d = pd.read_csv(CSV)
d["y"] = (d.label == "adversarial").astype(int)
u = d.drop_duplicates("text").reset_index(drop=True)
print(f"rows={len(d)}  distinct prompts={len(u)}  {u.groupby('label').size().to_dict()}\n")

def fit_predict(train, test):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)   # same settings as train_detector.py
    clf = LogisticRegression(class_weight="balanced", max_iter=2000, C=5.0)
    clf.fit(vec.fit_transform(train.text), train.y)
    return clf.predict(vec.transform(test.text))

def report(name, test, pred):
    tp = ((pred == 1) & (test.y == 1)).sum(); fn = ((pred == 0) & (test.y == 1)).sum()
    fp = ((pred == 1) & (test.y == 0)).sum(); tn = ((pred == 0) & (test.y == 0)).sum()
    print(f"{name:46s} n={len(test):3d}  attack recall={tp/max(tp+fn,1):.2f}  false-positive rate={fp/max(fp+tn,1):.2f}")

# 1. original-style split (with duplicates)
tr, te = train_test_split(d, test_size=0.25, random_state=42, stratify=d.y)
report("original split", te, fit_predict(tr, te))
print(f"  share of test prompts also present in train: {te.text.isin(tr.text).mean():.0%}")

# 2. 5-fold CV on distinct prompts, 10 seeds
rec, fpr = [], []
for seed in range(10):
    pred = np.zeros(len(u), int)
    for a, b in StratifiedKFold(5, shuffle=True, random_state=seed).split(u, u.y):
        pred[b] = fit_predict(u.iloc[a], u.iloc[b])
    rec.append(((pred == 1) & (u.y == 1)).sum() / (u.y == 1).sum())
    fpr.append(((pred == 1) & (u.y == 0)).sum() / (u.y == 0).sum())
print(f"distinct-prompt 5-fold CV (10 seeds): recall={np.mean(rec):.2f}  FPR={np.mean(fpr):.2f}\n")

# 3. cross-script
for a, b in (("roman_urdu", "urdu_script"), ("urdu_script", "roman_urdu")):
    te = u[u.script == b]; report(f"train {a} -> test {b}", te, fit_predict(u[u.script == a], te))

# 4. unseen attack category (+30% of benign held out)
for cat in ("injection", "jailbreak"):
    te = pd.concat([u[u.category == cat], u[u.category == "benign"].sample(frac=0.3, random_state=0)])
    report(f"unseen category: {cat}", te, fit_predict(u.drop(te.index), te))
